In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader



import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from Data.class_dataset import MRIDataset
from Model.model import build_vit3d

# --- Cargar datos de test ---

df=pd.read_csv("../Training/training_data.csv")

_, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
test_imgs = test_df["Path"].tolist()
test_ages = test_df["Age"].tolist()
test_dataset = MRIDataset(test_imgs, test_ages)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# --- Cargar modelo ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_vit3d()
state_dict = torch.load("../Training/Trained_models/model_6.pth", map_location=device)
# Si las claves tienen 'module.' al inicio, elimínalo
if any(k.startswith('module.') for k in state_dict.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_key = k.replace('module.', '', 1)
        new_state_dict[new_key] = v
    state_dict = new_state_dict
model.load_state_dict(state_dict)
model.to(device)
model.eval()



ViT3D(
  (patch_to_embedding): Linear(in_features=4096, out_features=1024, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=1024, out_features=1536, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=512, out_features=1024, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=1024, out_features=2048, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=

In [4]:
# --- Evaluar ---
all_preds = []
all_ages = []
with torch.no_grad():
    for imgs, ages in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs)
        all_preds.append(preds.cpu())
        all_ages.append(ages.unsqueeze(1).cpu())
all_preds = torch.cat(all_preds).numpy().flatten()
all_ages = torch.cat(all_ages).numpy().flatten()
all_paths = test_df["Path"].tolist()

mae = mean_absolute_error(all_ages, all_preds)
r2 = r2_score(all_ages, all_preds)
print(f"Test MAE: {mae:.2f}")
print(f"Test R2: {r2:.2f}")

Test MAE: 3.78
Test R2: 0.94


In [ ]:
#crear un df con las edades reales, predichas y el ID
results_df = pd.DataFrame({
    "ID": all_paths,
    "Age": all_ages,
    "Prediction": all_preds,
    "Error": all_preds - all_ages, 
    'Absolute Error': np.abs(all_preds - all_ages)
})
results_df['ID']=results_df['ID'].replace('/data/lautaro/quasiraw/', '', regex=True)
results_df['ID']=results_df['ID'].replace('.nii.gz', '', regex=True)
results_df.sort_values(by='Absolute Error', ascending=True, inplace=True)
results_df.to_csv("test_predictions_6.csv", index=False)

In [6]:
results_df

,ID,Age,Prediction,Error,Absolute Error
1172,sub-855574161779,19.0,19.000446,0.000446,0.000446
359,sub-611_ses-wave2,58.0,58.000679,0.000679,0.000679
1440,sub-CC510551,61.0,61.001469,0.001469,0.001469
891,PT030_OpenNeuro_ds000119_sub-54,13.0,13.004164,0.004164,0.004164
788,024_S_6472_I1016587,68.0,68.013588,0.013588,0.013588
...,...,...,...,...,...
343,PT030_OpenNeuro_ds005126_sub-05,59.0,35.509708,-23.490292,23.490292
1498,PT030_OpenNeuro_ds005126_sub-08,47.0,22.479292,-24.520708,24.520708
1433,PT030_OpenNeuro_ds004196_sub-02,36.0,61.031380,25.031380,25.031380
1306,PT030_OpenNeuro_ds002366_sub-22,56.0,84.557243,28.557243,28.557243
